# 🏔️ Open Lakehouse Demo

## ローカルSparkからSnowflake Managed Icebergテーブルを読み取る

### 概要
このデモでは、**Snowflake Horizon REST Catalog API** を使用して、ローカル環境で動作する Apache Spark から Snowflake 上の **Managed Iceberg テーブル** に直接アクセスします。

### Open Lakehouse とは？
Open Lakehouse は、データレイクの柔軟性とデータウェアハウスの信頼性を兼ね備えたアーキテクチャです。
Apache Iceberg のようなオープンテーブルフォーマットを採用することで、**ベンダーロックインを回避**し、様々なエンジンから同じデータにアクセスできます。

### このデモで学べること
1. **Horizon REST Catalog API** - Snowflakeが提供するIceberg REST Catalogエンドポイント
2. **Vended Credentials** - Snowflakeが一時的なクラウドストレージ認証情報を自動発行する仕組み
3. **マルチエンジンアクセス** - Snowflake以外のエンジン（Spark）からIcebergテーブルを読み取る方法

### アーキテクチャ図
```
┌─────────────────┐     REST API      ┌─────────────────────────┐
│  Local Spark    │ ◄──────────────► │  Snowflake Horizon      │
│  (このノートブック)  │                  │  REST Catalog API       │
└────────┬────────┘                  └───────────┬─────────────┘
         │                                       │
         │  Vended Credentials                   │ メタデータ管理
         │  (一時的なS3アクセス権)                  │
         ▼                                       ▼
┌─────────────────────────────────────────────────────────────┐
│                    Amazon S3                                │
│            (Icebergデータファイル: Parquet)                   │
└─────────────────────────────────────────────────────────────┘
```

## 1. 環境設定 (Java)

Apache Spark は Java で動作するため、適切な Java バージョンが必要です。
Iceberg 1.9.x は **Java 11以上** を必要とします。

> ⚠️ **注意**: macOS標準のJava 8では動作しません

In [ ]:
import os

# ===== Java環境の設定 =====
# conda環境内のJava 11を使用
JAVA_HOME = "/Users/kfukamori/opt/anaconda3/envs/spark-iceberg"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = f"{JAVA_HOME}/bin:" + os.environ["PATH"]

# Javaバージョンを確認（11以上であることを確認）
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java Version:")
print(result.stderr)

## 2. Snowflake 接続設定

Horizon REST Catalog API に接続するための設定を行います。

### 必要な情報
| 項目 | 説明 |
|------|------|
| **アカウント識別子** | `<account_locator>.<region>.<cloud>` 形式 |
| **ロール** | Icebergテーブルへのアクセス権を持つロール |
| **PAT (Programmatic Access Token)** | Snowsight → プロフィール → Programmatic Access Tokens で生成 |
| **ウェアハウス** | クエリ実行用のウェアハウス |

> 💡 **ヒント**: アカウント名にアンダースコア（_）が含まれる場合、アカウントロケーター形式を使用してください。
> Java のホスト名検証でアンダースコアが無効とみなされるためです。

In [ ]:
# ===== Snowflake 接続設定 =====

# アカウント識別子（アカウントロケーター形式: <locator>.<region>.<cloud>）
# ※ アンダースコアを含むアカウント名はJavaのSSL検証でエラーになるため、この形式を使用
SNOWFLAKE_ACCOUNT = "YOUR_ACCOUNT_LOCATOR.YOUR_REGION.aws"  # 例: zx48016.ap-northeast-1.aws

# 使用するロール（Icebergテーブルへの SELECT 権限が必要）
SNOWFLAKE_ROLE = "ICEBERG_SPARK_ROLE"

# 対象データベース（これがIceberg Catalogの "warehouse" として機能）
SNOWFLAKE_DATABASE = "ICEBERG_DEMO_DB"

# クエリ実行用ウェアハウス
SNOWFLAKE_WAREHOUSE = "COMPUTE_WH"

# S3のリージョン（External Volumeが存在するリージョン）
AWS_REGION = "ap-northeast-1"

# ===== PAT (Programmatic Access Token) =====
# Snowsight → 右上のプロフィール → Programmatic Access Tokens で生成
# ※ トークン生成時に上記のロールを指定すること
PAT_TOKEN = "YOUR_PAT_TOKEN_HERE"  # ここにPATトークンを貼り付け

# ===== 派生設定（自動計算） =====
# Horizon REST Catalog API のエンドポイント URL
HORIZON_CATALOG_URI = f"https://{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com/polaris/api/catalog"

# セッションスコープ（使用するロールを指定）
HORIZON_SESSION_ROLE = f"session:role:{SNOWFLAKE_ROLE}"

# SparkでのCatalog名（データベース名と同じにする）
CATALOG_NAME = SNOWFLAKE_DATABASE

# Icebergランタイムのバージョン
ICEBERG_VERSION = "1.9.1"

# 設定確認
print("=" * 50)
print("Snowflake 接続設定")
print("=" * 50)
print(f"アカウント    : {SNOWFLAKE_ACCOUNT}")
print(f"データベース  : {SNOWFLAKE_DATABASE}")
print(f"ロール        : {SNOWFLAKE_ROLE}")
print(f"ウェアハウス  : {SNOWFLAKE_WAREHOUSE}")
print(f"Catalog URI   : {HORIZON_CATALOG_URI}")

## 3. OAuth トークンの取得

Horizon REST Catalog API は **OAuth 2.0** で認証します。
PAT（Programmatic Access Token）を使って、OAuth アクセストークンを取得します。

### 認証フロー
```
PAT Token  ──────►  /oauth/tokens  ──────►  OAuth Access Token
(長期間有効)        (トークン交換)            (短期間有効、約1時間)
```

> 📝 **ポイント**: PATを直接使用するのではなく、OAuthトークンに交換する必要があります。
> これにより、短期間で失効するトークンを使用でき、セキュリティが向上します。

In [ ]:
import requests

# OAuth トークン交換エンドポイント
token_url = f"{HORIZON_CATALOG_URI}/v1/oauth/tokens"

# PATをOAuthアクセストークンに交換
response = requests.post(
    token_url,
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",    # OAuth 2.0 クライアント認証フロー
        "scope": HORIZON_SESSION_ROLE,          # 使用するロールをスコープで指定
        "client_secret": PAT_TOKEN              # PATをシークレットとして送信
    },
    verify=True
)

if response.status_code == 200:
    token_data = response.json()
    # 取得したOAuthトークンを保存（後続のSpark設定で使用）
    OAUTH_TOKEN = token_data["access_token"]
    print("✅ OAuth トークンを取得しました")
    print(f"   トークンタイプ: {token_data['token_type']}")
    print(f"   トークン（先頭50文字）: {OAUTH_TOKEN[:50]}...")
else:
    print(f"❌ OAuth トークンの取得に失敗: {response.status_code}")
    print(response.text)

## 4. Spark Session の作成

ここが最も重要な設定です。Apache Spark を **Iceberg REST Catalog** クライアントとして構成します。

### 主要な設定項目

| 設定 | 説明 |
|------|------|
| `type = rest` | Iceberg REST Catalog プロトコルを使用 |
| `uri` | Horizon REST Catalog API のエンドポイント |
| `token` | OAuth アクセストークン |
| `warehouse` | 対象データベース名 |
| `X-Iceberg-Access-Delegation: vended-credentials` | **Vended Credentials を有効化** |

### Vended Credentials とは？
従来、Sparkからクラウドストレージ（S3）にアクセスするには、AWSアクセスキーをSpark側で設定する必要がありました。
**Vended Credentials** を使うと、Snowflakeが**一時的なアクセス権**を発行するため、Spark側でのクラウド認証設定が不要になります。

```
Spark  ──►  Horizon API  ──►  一時S3認証情報を発行  ──►  Sparkがデータ読み取り
      (vended-credentials)        (約60分有効)
```

In [ ]:
from pyspark.sql import SparkSession

# 既存のSparkセッションがあれば停止
try:
    spark.stop()
except:
    pass

# ===== Spark Session の構築 =====
spark = (
    SparkSession.builder
    .appName("SnowflakeIcebergDemo")
    .master("local[*]")  # ローカルモード（全CPUコア使用）
    
    # ----- ネットワーク設定 -----
    # ローカル実行時のポート競合を回避
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    
    # ----- 依存JARの設定 -----
    # Iceberg Spark ランタイムと AWS S3 アクセス用バンドル
    .config(
        "spark.jars.packages",
        f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION},"
        f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}"
    )
    
    # ----- Iceberg SQL拡張 -----
    # SHOW NAMESPACES, SHOW TABLES などのIceberg固有SQLを有効化
    .config("spark.sql.extensions", 
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    
    # デフォルトカタログの設定
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    
    # ----- Horizon REST Catalog 設定 -----
    # Catalogの実装クラス
    .config(f"spark.sql.catalog.{CATALOG_NAME}", 
            "org.apache.iceberg.spark.SparkCatalog")
    
    # Catalog タイプ: REST（Iceberg REST Catalog プロトコル）
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    
    # Horizon REST Catalog API のエンドポイント
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", HORIZON_CATALOG_URI)
    
    # Warehouse（Snowflakeのデータベース名）
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", CATALOG_NAME)
    
    # OAuth トークン（先ほど取得したもの）
    .config(f"spark.sql.catalog.{CATALOG_NAME}.token", OAUTH_TOKEN)
    
    # S3リージョン
    .config(f"spark.sql.catalog.{CATALOG_NAME}.client.region", AWS_REGION)
    
    # ----- Snowflake 固有ヘッダー -----
    # クエリ実行用ウェアハウスの指定
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.X-Snowflake-Warehouse", 
            SNOWFLAKE_WAREHOUSE)
    
    # 【重要】Vended Credentials の有効化
    # これにより、Snowflakeが一時的なS3アクセス認証情報を発行
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation", 
            "vended-credentials")
    
    # ベクトル化読み取りを無効化（互換性のため）
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    
    .getOrCreate()
)

# ログレベルを設定（ERRORのみ表示）
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark Session を作成しました")
print(f"   Spark バージョン: {spark.version}")
print(f"   デフォルトカタログ: {CATALOG_NAME}")

## 5. カタログの探索

ここからは、Sparkを使ってSnowflake上のIcebergテーブルを探索します。
Iceberg REST Catalogプロトコルでは、以下の階層構造でデータが管理されます：

```
Catalog (= Database)
  └── Namespace (= Schema)
        └── Table
```

まずは、利用可能な**Namespace（スキーマ）**を確認しましょう。

In [ ]:
# Namespace（スキーマ）一覧を表示
# Snowflakeのスキーマが Iceberg の Namespace として表示される
print(f"📁 {CATALOG_NAME} 内の Namespace 一覧:")
spark.sql("SHOW NAMESPACES").show(truncate=False)

## 6. テーブル一覧

PUBLIC スキーマ内のIcebergテーブルを確認します。

> 📝 **注意**: Horizon REST Catalog API 経由では、**Snowflake Managed Iceberg テーブル**のみが表示されます。
> 通常のSnowflakeテーブル（Native Table）は表示されません。

In [ ]:
# PUBLIC スキーマ内のテーブル一覧
print(f"📋 {CATALOG_NAME}.PUBLIC 内のテーブル一覧:")
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.PUBLIC").show(truncate=False)

## 7. テーブル定義の確認

SALES_DATA テーブルのスキーマ（カラム定義）を確認します。

In [ ]:
# テーブル定義（スキーマ）を表示
print(f"🔍 {CATALOG_NAME}.PUBLIC.SALES_DATA のスキーマ:")
spark.sql(f"DESCRIBE TABLE {CATALOG_NAME}.PUBLIC.SALES_DATA").show(truncate=False)

## 8. データの読み取り（SELECT *）

いよいよ、**SparkからSnowflake Managed Icebergテーブルのデータを直接読み取り**ます。

### ここで何が起きているか？
1. Spark が Horizon REST API に対してテーブルメタデータをリクエスト
2. Snowflake がテーブルの Iceberg メタデータ（manifest ファイルの場所等）を返却
3. 同時に **Vended Credentials**（一時的なS3認証情報）も発行
4. Spark が S3 から直接 Parquet ファイルを読み取り

> 🎯 **重要**: データは Snowflake を経由せず、**Spark が S3 から直接読み取り**ます。
> Snowflake は「カタログ」として機能し、データの場所と認証情報を提供するだけです。

In [ ]:
# 全データを取得して表示
# ※ Spark SQL で Snowflake 上の Iceberg テーブルにクエリを実行
print(f"📊 {CATALOG_NAME}.PUBLIC.SALES_DATA の全データ:")
spark.sql(f"SELECT * FROM {CATALOG_NAME}.PUBLIC.SALES_DATA").show(truncate=False)

## 9. 集計クエリ（地域別売上分析）

Sparkの強力な分散処理能力を活かして、集計クエリを実行します。
この処理は完全にSpark側で実行され、Snowflakeのコンピュートリソースは使用しません。

In [ ]:
# 地域別の売上集計
# ※ 集計処理は Spark のローカルエンジンで実行
print("📈 地域別売上サマリー:")
spark.sql(f"""
    SELECT 
        REGION,
        COUNT(*) as order_count,
        SUM(AMOUNT) as total_sales,
        ROUND(AVG(AMOUNT), 2) as avg_sale
    FROM {CATALOG_NAME}.PUBLIC.SALES_DATA 
    GROUP BY REGION
    ORDER BY total_sales DESC
""").show(truncate=False)

## 10. DataFrame API の活用

Spark SQL だけでなく、**DataFrame API** を使ってプログラマティックにデータを操作することもできます。
DataFrame API は、Python/Scala/Java から型安全にデータ操作を行う方法を提供します。

In [ ]:
# DataFrame としてテーブルを読み込み
df = spark.table(f"{CATALOG_NAME}.PUBLIC.SALES_DATA")

# レコード数を表示
print(f"📊 総レコード数: {df.count()}")

# スキーマを表示
print("\n🔍 DataFrame スキーマ:")
df.printSchema()

# 売上金額の上位3件を表示（DataFrame API使用）
print("\n🏆 売上金額 Top 3:")
df.orderBy(df["AMOUNT"].desc()).limit(3).show(truncate=False)

## 11. クリーンアップ

デモが完了したら、Sparkセッションを停止してリソースを解放します。

### まとめ
このデモでは、以下を実現しました：

✅ **Horizon REST Catalog API** を使った外部エンジンからのアクセス  
✅ **OAuth 2.0** による安全な認証  
✅ **Vended Credentials** によるシームレスなS3アクセス  
✅ **Spark SQL / DataFrame API** によるデータ操作

### 次のステップ
- 他のIcebergクライアント（Trino, Flink等）からの接続を試す
- より大きなデータセットでのパフォーマンス検証
- Snowflake側でのテーブル更新とSparkからの読み取り確認

In [ ]:
# Spark セッションを停止
spark.stop()
print("✅ Spark Session を停止しました")
print("\n🎉 Open Lakehouse デモ完了！")